In [15]:
import pandas as pd
import glob
import os
import numpy as np

# Carpeta donde están los CSV mensuales
ruta = "C:/Users/hunte/Documents/Appv2/ML-SmartBids/annio/*.csv"

archivos = sorted(glob.glob(ruta))

print(f"Archivos encontrados: {len(archivos)}")

if len(archivos) == 0:
    raise FileNotFoundError(
        "No se encontraron archivos CSV en datos_licitaciones/"
    )

dataframes = []



for archivo in archivos:
    nombre = os.path.basename(archivo)

    # Ajusta esto si los nombres no contienen "11" / "12"
    if "11" in nombre or "12" in nombre:

        print("\n" + "=" * 100)
        print(nombre)
        print("=" * 100)

        temp = pd.read_csv(
            archivo,
            sep=";",
            encoding="cp1252",
            encoding_errors="replace",
            low_memory=False
        )

        print("Filas:", f"{len(temp):,}")

        for col in ["Cantidad Ofertada", "Valor Total Ofertado"]:

            if col in temp.columns:
                print(
                    f"{col}: "
                    f"{temp[col].isna().sum():,} NaN "
                    f"({temp[col].isna().mean()*100:.2f}%)"
                )
            else:
                print(f"{col}: NO EXISTE con ese nombre")



for archivo in archivos:

    nombre = os.path.basename(archivo)
    print(f"\nLeyendo: {nombre}")

    df_mes = pd.read_csv(
        archivo,
        sep=";",
        encoding="cp1252",
        encoding_errors="replace",
        low_memory=False
    )

    # Detectar únicamente columnas de texto
    columnas_texto = df_mes.select_dtypes(
        include=["object", "string"]
    ).columns

    # Si una celda contiene el carácter de reemplazo �,
    # convertir TODA esa celda a NaN
    for columna in columnas_texto:

        mascara = (
            df_mes[columna]
            .astype("string")
            .str.contains("�", regex=False, na=False)
        )

        cantidad = mascara.sum()

        if cantidad > 0:
            print(
                f"  ⚠ {columna}: "
                f"{cantidad:,} valores con problemas -> NaN"
            )

            df_mes.loc[mascara, columna] = np.nan

    print(f"  ✓ Filas: {len(df_mes):,}")
    print(f"  ✓ Columnas: {len(df_mes.columns)}")

    dataframes.append(df_mes)


# ============================================================
# UNIR TODOS LOS MESES
# ============================================================

df = pd.concat(
    dataframes,
    ignore_index=True
)

print("\n" + "=" * 60)
print("DATASET ANUAL")
print("=" * 60)

print(f"Archivos unidos: {len(dataframes)}")
print(f"Filas totales: {len(df):,}")
print(f"Columnas: {len(df.columns)}")
print(f"Dimensiones: {df.shape}")


# ============================================================
# GUARDAR
# ============================================================

df.to_feather("licitaciones_anual.feather")

print("\n✓ Guardado como:")
print("licitaciones_anual.feather")

Archivos encontrados: 12

lic_2024-11.csv
Filas: 530,356
Cantidad Ofertada: 320,761 NaN (60.48%)
Valor Total Ofertado: 320,761 NaN (60.48%)

lic_2024-12.csv
Filas: 359,662
Cantidad Ofertada: 251,143 NaN (69.83%)
Valor Total Ofertado: 251,143 NaN (69.83%)

Leyendo: lic_2024-01.csv
  ⚠ Descripcion linea Adquisicion: 9 valores con problemas -> NaN
  ⚠ DescripcionProveedor: 11 valores con problemas -> NaN
  ✓ Filas: 203,486
  ✓ Columnas: 105

Leyendo: lic_2024-02.csv
  ⚠ Descripcion linea Adquisicion: 4 valores con problemas -> NaN
  ⚠ DescripcionProveedor: 4 valores con problemas -> NaN
  ✓ Filas: 216,727
  ✓ Columnas: 105

Leyendo: lic_2024-03.csv
  ⚠ DescripcionProveedor: 9 valores con problemas -> NaN
  ✓ Filas: 220,947
  ✓ Columnas: 105

Leyendo: lic_2024-04.csv
  ⚠ DescripcionProveedor: 4 valores con problemas -> NaN
  ✓ Filas: 254,411
  ✓ Columnas: 105

Leyendo: lic_2024-05.csv
  ⚠ Descripcion linea Adquisicion: 1 valores con problemas -> NaN
  ⚠ DescripcionProveedor: 5 valores con 

ParserError: Error tokenizing data. C error: out of memory

In [ ]:
# Buscar columnas relacionadas con estado, adjudicación y fechas

import pandas as pd


archivo = r"C:/Users/hunte/Documents/Appv2/ML-SmartBids/annio/lic_2024-11.csv"

nov = pd.read_csv(
    archivo,
    sep=";",
    encoding="cp1252",
    encoding_errors="replace",
    low_memory=False
)


palabras = [
    "estado",
    "adjud",
    "fecha",
    "cierre",
    "licitacion",
    "licitación",
    "codigo",
    "código"
]

for palabra in palabras:
    encontrados = [
        col for col in nov.columns
        if palabra.lower() in col.lower()
    ]

    print(f"\n--- {palabra.upper()} ---")
    for col in encontrados:
        print(col)

ParserError: Error tokenizing data. C error: out of memory

In [ ]:
mask_sin_oferta = (
    nov["Cantidad Ofertada"].isna()
    & nov["Valor Total Ofertado"].isna()
)

print("=" * 80)
print("ESTADO - REGISTROS SIN OFERTA")
print("=" * 80)

print(
    nov.loc[mask_sin_oferta, "Estado"]
       .value_counts(dropna=False)
       .to_string()
)

print("\n" + "=" * 80)
print("ESTADO - REGISTROS CON OFERTA")
print("=" * 80)

print(
    nov.loc[~mask_sin_oferta, "Estado"]
       .value_counts(dropna=False)
       .to_string()
)

ESTADO - REGISTROS SIN OFERTA
Estado
Adjudicada                            298883
Revocada                                8217
Desierta (o art. 3 ó 9 Ley 19.886)      7703
Cerrada                                 5949
Suspendida                                 9

ESTADO - REGISTROS CON OFERTA
Estado
Adjudicada                            187137
Desierta (o art. 3 ó 9 Ley 19.886)     10947
Cerrada                                 5796
Revocada                                5699
Suspendida                                16


In [ ]:
mask_sin_oferta = (
    nov["Cantidad Ofertada"].isna()
    & nov["Valor Total Ofertado"].isna()
)

print("SIN DATOS DE OFERTA:")
print(
    nov.loc[mask_sin_oferta, "Estado Oferta"]
       .value_counts(dropna=False)
)

print("\nCON DATOS DE OFERTA:")
print(
    nov.loc[~mask_sin_oferta, "Estado Oferta"]
       .value_counts(dropna=False)
)

SIN DATOS DE OFERTA:
Estado Oferta
Aceptada     285037
Rechazada     35724
Name: count, dtype: int64

CON DATOS DE OFERTA:
Estado Oferta
Aceptada     186340
Rechazada     23255
Name: count, dtype: int64


In [ ]:
mask_sin_oferta = (
    nov["Cantidad Ofertada"].isna()
    & nov["Valor Total Ofertado"].isna()
)

columnas = [
    "Codigo",
    "CodigoExterno",
    "Codigoitem",
    "CodigoProveedor",
    "Estado",
    "Estado Oferta",
    "Cantidad",
    "Cantidad Ofertada",
    "Valor Total Ofertado",
    "CantidadAdjudicada",
    "MontoLineaAdjudica"
]

display(
    nov.loc[
        mask_sin_oferta,
        columnas
    ].head(30)
)

,Codigo,CodigoExterno,Codigoitem,CodigoProveedor,Estado,Estado Oferta,Cantidad,Cantidad Ofertada,Valor Total Ofertado,CantidadAdjudicada,MontoLineaAdjudica
5,9300375,2424-23-LE24,42384291,1787988.0,Cerrada,Aceptada,1,NaN,NaN,NaN,NaN
15,9304034,1966-3-O124,42462408,1879134.0,Cerrada,Aceptada,1,NaN,NaN,NaN,NaN
16,9304034,1966-3-O124,42462408,1385048.0,Cerrada,Aceptada,1,NaN,NaN,NaN,NaN
17,9304034,1966-3-O124,42462408,1713488.0,Cerrada,Aceptada,1,NaN,NaN,NaN,NaN
18,9304034,1966-3-O124,42462408,30723.0,Cerrada,Aceptada,1,NaN,NaN,NaN,NaN
19,9304034,1966-3-O124,42462408,18744.0,Cerrada,Aceptada,1,NaN,NaN,NaN,NaN
20,9304034,1966-3-O124,42462408,1510163.0,Cerrada,Aceptada,1,NaN,NaN,NaN,NaN
21,9304034,1966-3-O124,42462408,1812694.0,Cerrada,Aceptada,1,NaN,NaN,NaN,NaN
22,9304034,1966-3-O124,42462408,51411.0,Cerrada,Aceptada,1,NaN,NaN,NaN,NaN
23,9317524,1057417-54-LE24,41742797,NaN,Desierta (o art. 3 ó 9 Ley 19.886),Rechazada,2,NaN,NaN,NaN,NaN


In [ ]:
# NaN en ambas variables
mask_nan = (
    nov["Cantidad Ofertada"].isna()
    & nov["Valor Total Ofertado"].isna()
)

# Buscar licitaciones ADJUDICADAS que tengan NaN
codigos_nan_adjudicados = (
    nov.loc[
        mask_nan & (nov["Estado"] == "Adjudicada"),
        "CodigoExterno"
    ]
    .dropna()
    .unique()
)

print(
    "Licitaciones adjudicadas con algún registro NaN:",
    len(codigos_nan_adjudicados)
)

# Buscar una licitación que tenga AL MISMO TIEMPO
# registros con oferta y registros sin oferta
codigo_ejemplo = None

for codigo in codigos_nan_adjudicados:
    
    grupo = nov[nov["CodigoExterno"] == codigo]

    tiene_nan = (
        grupo["Cantidad Ofertada"].isna()
        & grupo["Valor Total Ofertado"].isna()
    ).any()

    tiene_datos = (
        grupo["Cantidad Ofertada"].notna()
        & grupo["Valor Total Ofertado"].notna()
    ).any()

    if tiene_nan and tiene_datos:
        codigo_ejemplo = codigo
        break

print("Ejemplo encontrado:", codigo_ejemplo)

Licitaciones adjudicadas con algún registro NaN: 2204
Ejemplo encontrado: 1075963-244-LR24


In [ ]:
codigo_ejemplo = "1075963-244-LR24"

caso = nov[
    nov["CodigoExterno"] == codigo_ejemplo
].copy()

columnas = [
    "CodigoExterno",
    "Codigoitem",
    "CodigoProveedor",
    "Estado",
    "Estado Oferta",
    "Cantidad",
    "Cantidad Ofertada",
    "Valor Total Ofertado",
    "CantidadAdjudicada",
    "MontoLineaAdjudica",
    "FechaAdjudicacion"
]

print("Total filas:", len(caso))
print("Ítems distintos:", caso["Codigoitem"].nunique())
print("Proveedores distintos:", caso["CodigoProveedor"].nunique())

print(
    "Filas sin oferta:",
    (
        caso["Cantidad Ofertada"].isna()
        & caso["Valor Total Ofertado"].isna()
    ).sum()
)

print(
    "Filas con oferta:",
    (
        caso["Cantidad Ofertada"].notna()
        & caso["Valor Total Ofertado"].notna()
    ).sum()
)

display(
    caso[columnas]
    .sort_values(["Codigoitem", "CodigoProveedor"])
)

Total filas: 572
Ítems distintos: 26
Proveedores distintos: 22
Filas sin oferta: 481
Filas con oferta: 91


,CodigoExterno,Codigoitem,CodigoProveedor,Estado,Estado Oferta,Cantidad,Cantidad Ofertada,Valor Total Ofertado,CantidadAdjudicada,MontoLineaAdjudica,FechaAdjudicacion
104,1075963-244-LR24,42471547,24141.0,Adjudicada,Aceptada,7000,NaN,NaN,NaN,NaN,2025-03-12
113,1075963-244-LR24,42471547,24786.0,Adjudicada,Aceptada,7000,NaN,NaN,NaN,NaN,2025-03-12
112,1075963-244-LR24,42471547,25418.0,Adjudicada,Aceptada,7000,NaN,NaN,NaN,NaN,2025-03-12
108,1075963-244-LR24,42471547,26931.0,Adjudicada,Aceptada,7000,NaN,NaN,NaN,NaN,2025-03-12
102,1075963-244-LR24,42471547,27380.0,Adjudicada,Aceptada,7000,NaN,NaN,NaN,NaN,2025-03-12
...,...,...,...,...,...,...,...,...,...,...,...
670,1075963-244-LR24,42471678,1419989.0,Adjudicada,Aceptada,300,NaN,NaN,NaN,NaN,2025-03-12
650,1075963-244-LR24,42471678,1640264.0,Adjudicada,Aceptada,300,NaN,NaN,NaN,NaN,2025-03-12
669,1075963-244-LR24,42471678,1793880.0,Adjudicada,Rechazada,300,NaN,NaN,NaN,NaN,2025-03-12
659,1075963-244-LR24,42471678,1867609.0,Adjudicada,Aceptada,300,NaN,NaN,NaN,NaN,2025-03-12


In [ ]:
caso["SinOferta"] = (
    caso["Cantidad Ofertada"].isna()
    & caso["Valor Total Ofertado"].isna()
)

resumen_items = (
    caso.groupby("Codigoitem")
    .agg(
        Filas=("Codigoitem", "size"),
        Proveedores=("CodigoProveedor", "nunique"),
        SinOferta=("SinOferta", "sum")
    )
)

resumen_items["ConOferta"] = (
    resumen_items["Filas"] - resumen_items["SinOferta"]
)

resumen_items["Porcentaje_NaN"] = (
    resumen_items["SinOferta"]
    / resumen_items["Filas"]
    * 100
)

display(resumen_items)

,Filas,Proveedores,SinOferta,ConOferta,Porcentaje_NaN
Codigoitem,,,,,
42471547,22,22,17,5,77.272727
42471550,22,22,20,2,90.909091
42471553,22,22,19,3,86.363636
42471557,22,22,18,4,81.818182
42471559,22,22,19,3,86.363636
42471594,22,22,17,5,77.272727
42471595,22,22,22,0,100.000000
42471598,22,22,14,8,63.636364
42471604,22,22,22,0,100.000000


In [ ]:
resumen_proveedores = (
    caso.groupby("CodigoProveedor")
    .agg(
        Filas=("CodigoProveedor", "size"),
        Items=("Codigoitem", "nunique"),
        SinOferta=("SinOferta", "sum")
    )
)

resumen_proveedores["ConOferta"] = (
    resumen_proveedores["Filas"]
    - resumen_proveedores["SinOferta"]
)

resumen_proveedores["Porcentaje_NaN"] = (
    resumen_proveedores["SinOferta"]
    / resumen_proveedores["Filas"]
    * 100
)

display(
    resumen_proveedores.sort_values(
        "Porcentaje_NaN",
        ascending=False
    )
)

,Filas,Items,SinOferta,ConOferta,Porcentaje_NaN
CodigoProveedor,,,,,
25418.0,26,26,25,1,96.153846
1014164.0,26,26,25,1,96.153846
266308.0,26,26,25,1,96.153846
58764.0,26,26,25,1,96.153846
90783.0,26,26,25,1,96.153846
62749.0,26,26,25,1,96.153846
54808.0,26,26,24,2,92.307692
1008780.0,26,26,24,2,92.307692
1978983.0,26,26,23,3,88.461538
